[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | Notebook

# How the Internet assigns and uses Autonomous Systems (ASes)

In [1]:
name = "Shivani Hariprasad"
date = "06/21/2026"

In [2]:
# Setup: shared utilities and tier definitions
import bz2
import urllib.request
import gzip
import io
import json
import zipfile
from contextlib import contextmanager
from pathlib import Path
from IPython.display import Markdown, display


@contextmanager
def open_safe(filename, encoding="utf-8"):
    """Open a file for reading, transparently handling .gz, .bz2, and .zip compression."""
    path = Path(filename)
    suffix = path.suffix.lower()
    if suffix == ".gz":
        with gzip.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".bz2":
        with bz2.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".zip":
        with zipfile.ZipFile(path) as zf:
            with zf.open(zf.namelist()[0]) as raw:
                yield io.TextIOWrapper(raw, encoding=encoding)
    else:
        with path.open(encoding=encoding) as f:
            yield f


TIERS = [
    ("edge",           1,     1),
    ("transit small",  2,     10),
    ("transit middle", 11,    1000),
    ("transit large",  1001,  10000),
    ("transit huge",   10001, -1),
]

---

## Task 2: ASN Classified by Customer Cone Size

Fill in the missing code in the cells below to produce **Table 1**.

### Data format

In `data/20260501.ppdc-ases.txt.bz2`, lines starting with `#` are comments.
All other lines start with an ASN followed by the ASNs in its customer cone.
The cone size for that ASN is the number of space-separated tokens on the line minus 1.

```
# This is a comment
# 23's customer cone size is 3 and includes (23, 4, 1)
23 23 4 1
# 1's customer cone size is 1 — it only includes itself
1 1
```

### Table 1 format

|           tier | range        | number of ASNs |   percentage |
| -------------: | ------------ | -------------: | -----------: |
|           edge | 1            |        [total] | [percentage] |
|  transit small | 2..10        |        [total] | [percentage] |
| transit middle | 11..1000     |        [total] | [percentage] |
|  transit large | 1001..10000  |        [total] | [percentage] |
|   transit huge | 10001..[max] |        [total] | [percentage] |

Replace `[max]` with the actual maximum cone size found in the data.
Percentages are formatted to one decimal place.

In [4]:
AS_CONE_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as-relationships/20260501.ppdc-ases.txt.bz2"
AS_CONE_PATH = Path("data/20260501.ppdc-ases.txt.bz2")

def classify(size: int) -> str:
    for label, lo, hi in TIERS:
        if hi == -1:               # open-ended upper bound
            if size >= lo:
                return label
        elif lo <= size <= hi:
            return label
    return "unknown"

print(f"Downloading {AS_CONE_URL} ...")
AS_CONE_PATH.parent.mkdir(parents=True, exist_ok=True)  # ensure 'data/' exists
urllib.request.urlretrieve(AS_CONE_URL, AS_CONE_PATH)
if not AS_CONE_PATH.exists(): 
    print (f"Unable to save file {AS_CONE_PATH}")
    exit() 

# Build a dict mapping each ASN to its customer cone size
asn_to_size = {} 
with open_safe(AS_CONE_PATH) as fin:
    for line in fin:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        tokens = line.split()
        asn = tokens[0]
        asn_to_size[asn] = len(tokens) - 1 
print(f"Number of ASNs: {len(asn_to_size)}")


total = len(asn_to_size)
max_size = max(asn_to_size.values()) if asn_to_size else 0

tier_counts = {label: 0 for label, _, _ in TIERS}
for size in asn_to_size.values():
    tier_counts[classify(size)] += 1

header = "| tier | range | number of ASNs | percentage |"
sep    = "| ---: | ----- | ---: | ---: |"
table_1_rows = [header, sep]

for label, lo, hi in TIERS:
    if hi == -1:
        range_str = f"{lo}+"
    elif lo == hi:
        range_str = f"{lo}"
    else:
        range_str = f"{lo}-{hi}"
    count = tier_counts[label]
    pct = (count / total * 100) if total else 0.0
    table_1_rows.append(f"| {label} | {range_str} | {count} | {pct:.1f}% |")
table_1 = "\n".join(table_1_rows)

Number of ASNs: 79644


In [5]:
display(Markdown(table_1))
Path("tables").mkdir(exist_ok=True)
Path("tables/asn-cone-tiers.md").write_text(table_1, encoding="utf-8")

| tier | range | number of ASNs | percentage |
| ---: | ----- | ---: | ---: |
| edge | 1 | 67090 | 84.2% |
| transit small | 2-10 | 9977 | 12.5% |
| transit middle | 11-1000 | 2511 | 3.2% |
| transit large | 1001-10000 | 55 | 0.1% |
| transit huge | 10001+ | 11 | 0.0% |

270

### Question 1

What percentage of ASNs are edge ASes (customer cone size of 1)? What does this suggest about the structure of the Internet?

About 84.2% of ASNs are edge ASes, showing that the Internet is highly hierarchical and skewed: the vast majority of networks are stubs that buy connectivity but provide transit to no one, while a small number of asns carry the bulk of the routing hierarchy.

### Question 2

How large is the maximum customer cone? What does this tell you about the most influential ASes on the Internet?

In [6]:
biggest_asn = max(asn_to_size, key=asn_to_size.get)
print(f"Max cone size: {max_size} ({max_size/total*100:.1f}% of all ASNs), held by AS{biggest_asn}")

Max cone size: 56415 (70.8% of all ASNs), held by AS3356


The largest customer cone is 56,415 ASes (70.8% of all ASNs), held by AS3356 (Lumen/Level 3), showing that the most influential ASes are a tiny set of tier-1 transit providers whose reach spans most of the routed Internet, giving them high structural control and reach.

### Question 3

How do the proportions of edge, small transit, and large transit ASes compare? What does this distribution reveal about how ASes are organized hierarchically?

Edge ASes massively dominate at 84.2%, small transit makes up 12.5%, and the larger transit tiers shrink rapidly to almost nothing (large transit is just 0.1% and huge transit is only 11 ASes), so the proportions fall off steeply as cone size grows. This reveals a clear hierarchical pyramid: a huge base of stub networks that only buy connectivity sits beneath progressively fewer transit providers, with a tiny apex of tier-1 ASes providing reach to nearly the entire Internet.

---

## Task 3: How Are ASN Tiers Divided Across Countries?

Fill in the missing code in the cells below to produce **Table 2**.

- Use the tier classification from Task 2 (`classify()` and `TIERS`).
- Use `data/as2org.jsonl` to map each ASN to its organization's 2-letter country code.
  Each line is a JSON object with a `"country"` field and a `"members"` list of ASN strings.
- Find the top 4 countries by total ASN count across all tiers.
  All other countries map to **other**.
- The top-4 country codes become the dynamic column headers (e.g. `US`, `CN`).

### Table 2 format

| name           | 1st           | 2nd           | 3rd           | 4th           | other         |
| -------------- | ------------- | ------------- | ------------- | ------------- | ------------- |
| edge           | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit small  | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit middle | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit large  | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit huge   | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |

Replace `1st`/`2nd`/`3rd`/`4th` with the actual 2-letter country codes.
Each cell shows `total (X.X%)`, where `X.X%` is that country's share of the tier's total.

In [7]:
AS2ORG_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as2org/as2org.jsonl"
AS2ORG_PATH = Path("data/as2org.jsonl")

print(f"Downloading {AS2ORG_URL} ...")
AS2ORG_PATH.parent.mkdir(parents=True, exist_ok=True)  # ensure 'data/' exists
urllib.request.urlretrieve(AS2ORG_URL, AS2ORG_PATH)
if not AS2ORG_PATH.exists(): 
    print (f"Unable to save file {AS2ORG_PATH}")
    exit() 
    
asn_to_country = {}
with open_safe(AS2ORG_PATH) as fin:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        # Parse the JSON line, extract the ASNs in the members list
        # and country code, and add it to the asn_to_country dict.
        obj = json.loads(line)
        country = obj.get("country")
        for asn in obj.get("members", []):
            asn_to_country[asn] = country
print(f"number of ASNs with countries: {len(asn_to_country)}")

number of ASNs with countries: 120793


In [9]:
from collections import defaultdict

# Count ASNs per (tier, country) pair
tier_country_counts = defaultdict(lambda: defaultdict(int))
country_totals = defaultdict(int)
for asn, size in asn_to_size.items():
    tier = classify(size)
    country = asn_to_country.get(asn, "unknown")
    tier_country_counts[tier][country] += 1
    country_totals[country] += 1

# Find top-4 countries by total ASN count
top4 = [c for c, _ in sorted(country_totals.items(), key=lambda x: -x[1])[:4]]
columns = top4 + ["other"]

# Build Markdown table
header = "| tier | " + " | ".join(columns) + " |"
sep    = "| --- | " + " | ".join(["---"] * len(columns)) + " |"
table_2_rows = [header, sep]

# For each tier, build a row with each country's count and share of the tier total
for label, _, _ in TIERS:
    counts = tier_country_counts[label]
    tier_total = sum(counts.values())
    cells = []
    for col in columns:
        if col == "other":
            n = sum(v for c, v in counts.items() if c not in top4)
        else:
            n = counts.get(col, 0)
        pct = (n / tier_total * 100) if tier_total else 0.0
        cells.append(f"{n} ({pct:.1f}%)")
    table_2_rows.append(f"| {label} | " + " | ".join(cells) + " |")

table_2 = "\n".join(table_2_rows)

In [10]:
display(Markdown(table_2))

| tier | US | BR | RU | IN | other |
| --- | --- | --- | --- | --- | --- |
| edge | 16587 (24.7%) | 5943 (8.9%) | 4140 (6.2%) | 2475 (3.7%) | 37945 (56.6%) |
| transit small | 1582 (15.9%) | 1901 (19.1%) | 688 (6.9%) | 315 (3.2%) | 5491 (55.0%) |
| transit middle | 414 (16.5%) | 350 (13.9%) | 127 (5.1%) | 66 (2.6%) | 1554 (61.9%) |
| transit large | 10 (18.2%) | 7 (12.7%) | 8 (14.5%) | 3 (5.5%) | 27 (49.1%) |
| transit huge | 7 (63.6%) | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) | 4 (36.4%) |

In [11]:
table_2

'| tier | US | BR | RU | IN | other |\n| --- | --- | --- | --- | --- | --- |\n| edge | 16587 (24.7%) | 5943 (8.9%) | 4140 (6.2%) | 2475 (3.7%) | 37945 (56.6%) |\n| transit small | 1582 (15.9%) | 1901 (19.1%) | 688 (6.9%) | 315 (3.2%) | 5491 (55.0%) |\n| transit middle | 414 (16.5%) | 350 (13.9%) | 127 (5.1%) | 66 (2.6%) | 1554 (61.9%) |\n| transit large | 10 (18.2%) | 7 (12.7%) | 8 (14.5%) | 3 (5.5%) | 27 (49.1%) |\n| transit huge | 7 (63.6%) | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) | 4 (36.4%) |'

### Question 4

Which countries have the most transit huge ASNs? What does this tell you about where Internet infrastructure is concentrated?

In [12]:
# Which countries hold the transit-huge ASes?
huge_by_country = tier_country_counts["transit huge"]
for country, n in sorted(huge_by_country.items(), key=lambda x: -x[1]):
    print(f"{country}: {n}")

# And the actual ASNs, for a sanity check
print("\nASNs in transit huge:")
for asn, size in sorted(asn_to_size.items(), key=lambda x: -x[1]):
    if classify(size) == "transit huge":
        print(f"AS{asn}  cone={size}  country={asn_to_country.get(asn, 'unknown')}")

US: 7
SE: 1
IT: 1
HK: 1
GB: 1

ASNs in transit huge:
AS3356  cone=56415  country=US
AS1299  cone=41730  country=SE
AS3257  cone=41724  country=US
AS174  cone=41504  country=US
AS2914  cone=24384  country=US
AS6939  cone=22203  country=US
AS6453  cone=22014  country=US
AS6461  cone=20314  country=US
AS6762  cone=18860  country=IT
AS3491  cone=15773  country=HK
AS9002  cone=12465  country=GB


The US dominates the transit huge tier with 7 of the 11 largest ASes (including AS3356, AS174, and AS3257), with Sweden, Italy, Hong Kong, and the UK holding one each, showing that the Internet's top-level transit infrastructure is heavily concentrated in the United States, reflecting where the major tier-1 providers are headquartered.

### Question 5

What proportion of all ASNs fall in the "other" category? What does this suggest about the geographic distribution of the global Internet?

Roughly 56.5% of all ASNs fall outside the top 4 countries, showing that while top-level transit is US-centric, the broad base of the global Internet is widely distributed across many countries, each contributing a small share of the world's networks.

### Question 6

Do the same countries dominate across all AS tiers (edge, transit small, transit huge)? What patterns do you observe across the rows?

In [14]:
# Per-tier share for each top-4 country + US trend across tiers
for label, _, _ in TIERS:
    counts = tier_country_counts[label]
    total = sum(counts.values())
    row = {c: counts.get(c, 0) for c in top4}
    pcts = {c: (row[c] / total * 100 if total else 0) for c in top4}
    print(f"{label:14} total={total:6}  " +
          "  ".join(f"{c}={row[c]}({pcts[c]:.1f}%)" for c in top4))

edge           total= 67090  US=16587(24.7%)  BR=5943(8.9%)  RU=4140(6.2%)  IN=2475(3.7%)
transit small  total=  9977  US=1582(15.9%)  BR=1901(19.1%)  RU=688(6.9%)  IN=315(3.2%)
transit middle total=  2511  US=414(16.5%)  BR=350(13.9%)  RU=127(5.1%)  IN=66(2.6%)
transit large  total=    55  US=10(18.2%)  BR=7(12.7%)  RU=8(14.5%)  IN=3(5.5%)
transit huge   total=    11  US=7(63.6%)  BR=0(0.0%)  RU=0(0.0%)  IN=0(0.0%)


No, the same countries do not dominate uniformly: the US is consistently present and increasingly dominant toward the top tiers (peaking at 63.6% of transit huge), whereas countries like Brazil have many edge and mid-transit networks but no presence at the apex, showing that lower tiers are more geographically diverse while top-level transit infrastructure concentrates heavily in the US.